# Basic nearest neighbor/C2C algorithm

This notebook demonstrates a basic standard cloud-to-cloud (C2C) distance workflow in `py4dgeo`.

**Implemented by**  
Ronald Tabernig ([\@tabernig](https://github.com/tabernig))

First, import the required packages and helpers:

In [1]:
import py4dgeo
import numpy as np
from py4dgeo.util import find_file

Numba is not installed.
Use `pip install numba` to install it to enable faster computations in Vapc.
You can run py4dgeo.Vapc without Numba, but it may be slower.


Load a small pair of demo point clouds from the same test data used in the other tutorials:

In [2]:
# Fetch original test data
test_filename = "trier_sim.zip"
original_file = find_file(test_filename)
print(f"File downloaded to: {original_file}")

File downloaded to: C:\Users\nc298\AppData\Local\py4dgeo\py4dgeo\Cache\trier_sim.zip


In [3]:
epoch0 = py4dgeo.epoch.read_from_las(
    find_file("trier_sim_epoch_0.laz")
)
epoch1 = py4dgeo.epoch.read_from_las(
    find_file("trier_sim_epoch_1.laz")
)

[2026-04-22 14:23:51][INFO] Reading point cloud from file 'C:\Users\nc298\AppData\Local\py4dgeo\py4dgeo\Cache\trier_sim.zip.unzip\trier_sim_epoch_0.laz'
[2026-04-22 14:23:51][INFO] Reading point cloud from file 'C:\Users\nc298\AppData\Local\py4dgeo\py4dgeo\Cache\trier_sim.zip.unzip\trier_sim_epoch_1.laz'


Instantiate the minimal `C2C` method and run it on a small corepoint subset:

In [ ]:
corepoints = epoch0.cloud[::25]

c2c = py4dgeo.C2C(
    epochs=(epoch0, epoch1),
    corepoints=corepoints,
    max_distance=3.0,
)

distances = c2c.run()

In [5]:
distances[:10], np.nanmean(distances)

(array([0.00172307, 0.00152707, 0.01252007, 0.00784266, 0.00019976,
        0.00620458, 0.01018134, 0.00326379, 0.0069323 , 0.00600809]),
 np.float64(0.040701532424228595))

You can optionally enforce mutual matches using `correspondence_filter="mutual_nearest_neighbors"`. Non-mutual matches are kept in place and set to `np.nan`.

In [6]:
c2c_mnn = py4dgeo.C2C(
    epochs=(epoch0, epoch1),
    corepoints=corepoints,
    max_distance=3.0,
    correspondence_filter="mutual_nearest_neighbors",
)

distances_mnn = c2c_mnn.run()
np.sum(np.isnan(distances_mnn)), distances_mnn[:10]

(np.int64(54579),
 array([0.00172307, 0.00152707, 0.01252007, 0.00784266, 0.00019976,
        0.00620458, 0.01018134, 0.00326379, 0.0069323 , 0.00600809]))

In [7]:
outputfilepath = "c2c_results.las"

py4dgeo.write_c2c_results_to_las(
    outputfilepath,
    c2c,
    attribute_dict={"distances_C2C": distances,
                    "distances_C2C_MNN": distances_mnn},
)